# DDC without MATLAB files - native NumPy input

The numerical DDC core operates on `DDCDataset`; a MAT file is only an I/O boundary. This notebook builds the dataset directly from NumPy arrays, which is the preferred integration path for a Python-native upstream pipeline.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ddc import DDCDataset, DDCConfig, run_ddc

## 1. Build a small supported synthetic dataset

The fixture uses float64 frames, so it deliberately avoids the preserved MATLAB integer-frame defects.

In [ ]:
frames = np.concatenate((np.ones(11), [2.0], np.full(11, 4.0))).astype(np.float64)
x = np.concatenate((np.arange(11.0)*10, [0.2], np.arange(11.0)*10 + 0.5))
locations = np.column_stack((x, np.zeros(x.size))).astype(np.float64)
photons = np.ones(frames.size, dtype=np.float64)

dataset = DDCDataset(
    localizations=[locations],
    frames=[frames],
    photons=[photons],
)

print("n_images:", dataset.n_images)
print("locations:", dataset.localizations[0].shape, dataset.localizations[0].dtype)
print("frames:", dataset.frames[0].shape, dataset.frames[0].dtype)

## 2. Configure and run

In [ ]:
config = DDCConfig(
    resolution=10.0,
    stepper=12,
    n_f=2,
    photon_weighted_correction=0,
    timer=30.0,
)
rngs = [np.random.default_rng(4)]
result = run_ddc(dataset, config, rngs=rngs)

## 3. Inspect the result

In [ ]:
state = result.states[0]
print("steps:", result.steps)
print("final shape:", result.final_localizations[0].shape)
print("likelihood:", state.likelihood)
print("number history:", state.number_of_localizations)
print("trajectory shape:", result.trajectories[0].shape)

## 4. Integrate arrays from your own Python pipeline

Replace `locations`, `frames`, and optional `photons` with arrays from your upstream code. For real data, omit `true_localizations` (or pass `None` or `[]`); it is used only for simulation diagnostics. Important requirements:

- `locations.shape == (n, d)`;
- one frame value for each localization row;
- one photon value per localization when photon-weighted correction is enabled;
- keep floating frame dtype for the normal supported path if you want to avoid the preserved integer-frame MATLAB defects;
- for multiple images, store one NumPy array per image in each list.

In [ ]:
# Example for multiple Python-native images:
# dataset = DDCDataset(
#     localizations=[loc_img0, loc_img1, ...],
#     frames=[frames_img0, frames_img1, ...],
#     photons=[photons_img0, photons_img1, ...],
# )
# children = np.random.SeedSequence(123).spawn(dataset.n_images)
# rngs = [np.random.default_rng(child) for child in children]
# result = run_ddc(dataset, DDCConfig(), rngs=rngs)